# 第 3 章 §3.3（Google ADK 版）—— 解读

> 书里 ADK 并行实现的注释版。**代码未做适配**（原样保留），只在关键处插入中文解读。
>
> 与 LangChain 版不同，这份是**代码片段**而非可运行 notebook。下面说明原因和结构。

## 它在讲什么

一段声明式代码，搭出这样的流水线：

```
SequentialAgent  (先并行，后汇总)
  ├── ParallelAgent            ← 三个研究员【同时】工作
  │     ├── 可再生能源研究员   (google_search)
  │     ├── 电动车研究员       (google_search)
  │     └── 碳捕获研究员       (google_search)
  └── SynthesisAgent           ← 读 state，合成一份带出处的报告
```

**三个研究员之间没有任何数据依赖**，所以能并行；汇总必须等它们全部完成，所以串行。
这就是并行模式的标准形状：**扇出 → 汇合**。

## 为什么这份不能直接跑

| 缺失 | 说明 |
|---|---|
| `import` | 片段里没有 import（原文交代了要放进 `agent.py`） |
| `Runner` / `Session` | ADK 把执行循环收走了，需要 `InMemoryRunner` 之类的宿主 |
| `GEMINI_MODEL` 的值 | 书里写 `gemini-2.0-flash`，该模型已下线 |

ADK 的设计取向是：**你声明结构，框架负责执行**。
所以它的示例天然是"片段 + 宿主"的形式，而不是自包含脚本。

对照 §2.4 那份可以独立跑的 ADK routing 示例——那份之所以能跑，
是因为我们补了 `InMemoryRunner` 和 session 创建。

## 两个实测发现

**① `ParallelAgent` / `SequentialAgent` 已被标记弃用**

在 ADK 2.13.0 上构造它们会打印：

```
DeprecationWarning: ParallelAgent is deprecated in favor of Workflow
                   and will be removed in a future version.
```

官方建议迁移到 `google.adk.workflow.Workflow`（graph-based workflow node）。
书里的写法**目前仍能工作**，但新项目建议直接用 `Workflow`。

**② 免费层几乎跑不动这一节**

三个研究员各自都要多轮（决定搜什么 → 搜索 → 总结），一次扇出实际是
6~9 次模型调用。Google 免费层限制是 **5 请求/分钟**（按模型分桶）。

> 这也是第 2 章 §2.4 里我们撞 429 的同一个根因。
> **并行的第一成本不是 CPU，是配额** —— 这一节是这句话最好的例证。

In [ ]:
# =============================================================================
# 第 3 章 §3.3（Google ADK 版）—— 注释解读
# =============================================================================
#
# ⚠️ 先看清这份文件的【性质】：它是一个**代码片段**，不是能直接跑的 notebook。
#
#   证据：没有 import、没有 Runner、没有 main()。
#   原文第一行就写了 "Part of agent.py --> Follow .../quickstart/"，
#   意思是这段要放进 ADK 项目的 `agent.py`，再用 `adk web` / `adk run` 启动。
#
#   对照：LangChain 版是完整可跑的；ADK 版是"贴进框架里才活"的骨架。
#   这是 ADK 的设计取向——它把执行循环（Runner / Session / Event）收走了，
#   你只声明"有哪些 agent、什么顺序"。
#
# 本 notebook 只做解读，不执行（原因见文末）。
#
# ⚠️ 另一个实测发现：ParallelAgent / SequentialAgent 在 ADK 2.13 里
#    **已标记弃用**，官方建议迁移到 `Workflow`（graph-based workflow node）。
#    书里的写法目前仍能构造、仍能工作，但会打印 DeprecationWarning。
#
# =============================================================================


# Part of agent.py --> Follow https://google.github.io/adk-docs/get-started/quickstart/ to learn the setup
 # ─────────────────────────────────────────────────────────────────────────
 # ① 三个"研究员"子 agent —— 它们是并行的【单位】
 # ─────────────────────────────────────────────────────────────────────────
 # 每个 LlmAgent 都是独立的 agent：有自己的 instruction、自己的工具、自己的模型调用。
 # 三个之间**没有任何数据依赖**（各自研究各自的主题），所以能并行。
 #
 # ★ 关键机制一：output_key
 #   `output_key="renewable_energy_result"` 的意思是：
 #   这个 agent 的最终输出，会被写进【session state】的同名键里。
 #   这就是 agent 之间传递数据的唯一通道 —— 不是函数返回值，是共享状态。
 #
 # ★ 关键机制二：tools=[google_search]
 #   给了 agent 联网搜索能力。注意这会让每个 agent 变成"多轮"：
 #   先决定搜什么 → 搜 → 再总结。所以一次并行不是 3 次模型调用，
 #   而可能是 6~9 次（每个 agent 至少 2 轮）。
 #   免费层 5 RPM 的情况下，这一节基本跑不动。

 # --- 1. Define Researcher Sub-Agents (to run in parallel) ---

 # Researcher 1: Renewable Energy
 researcher_agent_1 = LlmAgent(
     name="RenewableEnergyResearcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in energy.
 Research the latest advancements in 'renewable energy sources'.
 Use the Google Search tool provided.
 Summarize your key findings concisely (1-2 sentences).
 Output *only* the summary.
 """,
     description="Researches renewable energy sources.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="renewable_energy_result"
 )

 # Researcher 2: Electric Vehicles
 researcher_agent_2 = LlmAgent(
     name="EVResearcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in transportation.
 Research the latest developments in 'electric vehicle technology'.
 Use the Google Search tool provided.
 Summarize your key findings concisely (1-2 sentences).
 Output *only* the summary.
 """,
     description="Researches electric vehicle technology.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="ev_technology_result"
 )

 # Researcher 3: Carbon Capture
 researcher_agent_3 = LlmAgent(
     name="CarbonCaptureResearcher",
     model=GEMINI_MODEL,
     instruction="""You are an AI Research Assistant specializing in climate solutions.
 Research the current state of 'carbon capture methods'.
 Use the Google Search tool provided.
 Summarize your key findings concisely (1-2 sentences).
 Output *only* the summary.
 """,
     description="Researches carbon capture methods.",
     tools=[google_search],
     # Store result in state for the merger agent
     output_key="carbon_capture_result"
 )

 # ─────────────────────────────────────────────────────────────────────────
 # ② ParallelAgent —— 并行的【声明式表达】
 # ─────────────────────────────────────────────────────────────────────────
 # 它本身不做任何计算，只做调度：把 sub_agents 同时启动，
 # 等**全部完成**后才算自己完成（fan-in 语义）。
 #
 # 这正是 LangChain 里 RunnableParallel 的对应物，区别在于：
 #   LangChain ：并行的单位是"链"（输入输出都在内存里流动）
 #   ADK       ：并行的单位是"agent"（各自有独立的多轮推理，结果落到 session state）
 #
 # 换句话说，ADK 的并行粒度更大 —— 是"多个完整 agent 同时工作"，
 # 而 LangChain 那版是"多个 prompt 同时发出去"。

 # --- 2. Create the ParallelAgent (Runs researchers concurrently) ---
 # This agent orchestrates the concurrent execution of the researchers.
 # It finishes once all researchers have completed and stored their results in state.
 parallel_research_agent = ParallelAgent(
     name="ParallelWebResearchAgent",
     sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
     description="Runs multiple research agents in parallel to gather information."
 )

 # ─────────────────────────────────────────────────────────────────────────
 # ③ 汇总 agent —— 汇合点（注定串行）
 # ─────────────────────────────────────────────────────────────────────────
 # ★ 关键机制三：instruction 里的 {renewable_energy_result} 这类占位符
 #   会被 ADK 用 session state 里同名键的值替换掉。
 #
 #   也就是说：三个研究员通过 output_key 写入 state，
 #   汇总 agent 通过 {占位符} 从 state 读取 —— 两者靠【字符串名字】约定，
 #   和 LangChain 里 RunnableParallel 的字典键名是同一类契约。
 #
 #   代价也一样：名字写错不会编译报错，只会在运行时填不进去。
 #   这正是第 2 章讲的"交接边界无校验"问题的又一次出现。
 #
 # ★ 另外注意这条 instruction 里的强制约束：
 #   "MUST be grounded *exclusively* on the information provided"
 #   这是在抑制模型自行补充外部知识（幻觉）。属于第 18 章 guardrails 的手段。

 # --- 3. Define the Merger Agent (Runs *after* the parallel agents) ---
 # This agent takes the results stored in the session state by the parallel agents
 # and synthesizes them into a single, structured response with attributions.
 merger_agent = LlmAgent(
     name="SynthesisAgent",
     model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
     instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

 Your primary task is to synthesize the following research summaries, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

 **Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the 'Input Summaries' below. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

 **Input Summaries:**

 *   **Renewable Energy:**
     {renewable_energy_result}

 *   **Electric Vehicles:**
     {ev_technology_result}

 *   **Carbon Capture:**
     {carbon_capture_result}

 **Output Format:**

 ## Summary of Recent Sustainable Technology Advancements

 ### Renewable Energy Findings
 (Based on RenewableEnergyResearcher's findings)
 [Synthesize and elaborate *only* on the renewable energy input summary provided above.]

 ### Electric Vehicle Findings
 (Based on EVResearcher's findings)
 [Synthesize and elaborate *only* on the EV input summary provided above.]

 ### Carbon Capture Findings
 (Based on CarbonCaptureResearcher's findings)
 [Synthesize and elaborate *only* on the carbon capture input summary provided above.]

 ### Overall Conclusion
 [Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

 Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
 """,
     description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
     # No tools needed for merging
     # No output_key needed here, as its direct response is the final output of the sequence
 )


 # ─────────────────────────────────────────────────────────────────────────
 # ④ SequentialAgent —— 把"并行"和"串行"拼成一条流水线
 # ─────────────────────────────────────────────────────────────────────────
 # sub_agents=[parallel_research_agent, merger_agent] 表示：
 #   先跑完 ParallelAgent（三个研究员并发），再跑 merger。
 #
 # 这就是并行模式的完整形状：**扇出 → 汇合**，
 # 只不过这里用两个容器 agent 把它包成了声明式结构：
 #
 #     SequentialAgent
 #       ├── ParallelAgent
 #       │     ├── LlmAgent(可再生能源)
 #       │     ├── LlmAgent(电动车)
 #       │     └── LlmAgent(碳捕获)
 #       └── LlmAgent(汇总)          ← 通过 {占位符} 读 state
 #
 # root_agent 这个名字是 ADK 的约定：adk web / adk run 会去找它作为入口。

 # --- 4. Create the SequentialAgent (Orchestrates the overall flow) ---
 # This is the main agent that will be run. It first executes the ParallelAgent
 # to populate the state, and then executes the MergerAgent to produce the final output.
 sequential_pipeline_agent = SequentialAgent(
     name="ResearchAndSynthesisPipeline",
     # Run parallel research first, then merge
     sub_agents=[parallel_research_agent, merger_agent],
     description="Coordinates parallel research and synthesizes the results."
 )

 root_agent = sequential_pipeline_agent

## 三个机制串起来看

ADK 的 agent 之间**不通过函数返回值传递数据**，而是通过 **session state**：

```
研究员 A ──output_key="renewable_energy_result"──┐
研究员 B ──output_key="ev_technology_result"─────┼──→ session state
研究员 C ──output_key="carbon_capture_result"────┘         │
                                                            ↓
汇总 agent 的 instruction 里写 {renewable_energy_result}
                ↓
        ADK 用 state 里的值替换占位符
```

**这个契约非常脆**：

- 键名写错 → 运行时占位符填不进去（不是编译错误）
- 三个研究员同时写 state → 如果键名撞了，会互相覆盖
- state 里有什么，取决于**执行顺序**——这正是为什么外层要包 `SequentialAgent`

这和 LangChain 版 `RunnableParallel` 的字典键名是同一类问题：
**靠字符串名字维系的数据契约，没有任何静态校验。**

## 和 LangChain 版的对照

| | LangChain（§3.3 上半） | ADK（§3.3 下半） |
|---|---|---|
| 并行的单位 | 一条链（一次 prompt + 一次调用） | **一个完整 agent**（多轮推理 + 工具） |
| 数据传递 | 内存里的 dict | **session state** |
| 契约形式 | `RunnableParallel` 的字典键名 | `output_key` ↔ `{占位符}` |
| 汇合方式 | `map_chain \| synthesis_prompt` | `SequentialAgent` 的第二段 |
| 能否独立跑 | ✅ 自包含 | ❌ 需要宿主 |
| 代码量 | 少 | 多（但结构更显式） |

**选型感觉**：

- 任务简单、一次调用能完成 → LangChain 的 `RunnableParallel` 更轻
- 需要多轮推理 + 工具、或者任务本身就是一个 agent → ADK 的粒度更合适

## 如果想在本机跑它

需要三步（都不适合在免费层做）：

```bash
# 1. 装 ADK（本机已在独立的 ~/agentic_adk_venv 里）
~/agentic_adk_venv/bin/pip list | grep google-adk

# 2. 把片段放进一个 agent 包
#    my_agent/agent.py   ← 这里放 root_agent
#    my_agent/__init__.py

# 3. 用 CLI 跑
adk run my_agent          # 或 adk web
```

⚠️ `google_search` 工具依赖 Google 的搜索接地（grounding），
需要可用的 Gemini API key；且免费层 5 RPM 撑不住 3 个多轮 agent 并发。